# Essai court LoRA Qwen sur Mac ou CUDA

Le notebook utilise le dépôt et le profil `environments/lora` déjà préparés. Il reconstruit ou vérifie les 4 000 exemples SFT, puis entraîne un seul adaptateur sur 64 exemples avec un lot physique de 2 et huit accumulations. Un contrôle des deux séquences les plus longues suit l'essai.

Sur Mac, ouvrir le notebook depuis le projet avec le noyau **TuniScope (notebooks)**. Sur Kaggle, exécuter d'abord la cellule de démarrage fournie séparément dans la conversation. Le même code utilise la révision Qwen officielle, les consignes v2 et le corpus SFT figé.


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import uuid
from zipfile import ZipFile, ZIP_DEFLATED

explicit = os.environ.get('TUNISCOPE_PROJECT_ROOT')
candidates = [Path(explicit)] if explicit else [Path.cwd(), *Path.cwd().parents]
ROOT = next((path.resolve() for path in candidates if (path / 'scripts/run_qwen_lora_smoke.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ouvrir depuis le dépôt à jour ou définir TUNISCOPE_PROJECT_ROOT.')

MODE = os.environ.get('TUNISCOPE_LORA_MODE', 'smoke')
DEVICE = os.environ.get('TUNISCOPE_LORA_DEVICE', 'auto')
if MODE not in {'smoke', 'off'} or DEVICE not in {'auto', 'cpu', 'mps', 'cuda'}:
    raise ValueError('Vérifier le mode et l’appareil demandés.')

SFT = ROOT / 'outputs/qwen/sft_v1_2026-11-07'
RUN = ROOT / 'outputs/qwen/notebooks' / uuid.uuid4().hex
MAC = {'post_warmup_step_seconds': 4.760580805331604, 'estimated_epoch_seconds': 1190.145201332901, 'dataset_sha256': '24b2f49d729dee65bb6e3c77269d473b2b9ddf967df2c66bd5b0ef8945740b68', 'script_sha256': '00ac8d1c12391bde911828af394d26ebaf8323076587615c8061b5739db5a873', 'lock_sha256': '73094b124d65c2eeb5c9316c9cd6946d7c523d8ea264e6812313bb7908bfd672', 'model_sha256': 'f47f71177f32bcd101b7573ec9171e6a57f4f4d31148d38e382306f42996874b', 'config_sha256': 'd11f8dd67a67874d355e872502f052d920b2ff6a9a639630404e7fc666330f4c'}

def run_script(script, *args):
    command = ['sh', str(ROOT / 'scripts/lora_uv.sh'), 'run', '--locked', '--offline',
               'python', '-u', str(ROOT / 'scripts' / script), *map(str, args)]
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

print('Projet', ROOT)
print('Mode', MODE, '| appareil', DEVICE)


## 1. Reproduire le corpus SFT

Le script vérifie les sources et reconstruit les mêmes exemples à partir des données locales préparées. L'empreinte doit correspondre au corpus du Mac.


In [ ]:
if MODE == 'smoke':
    option = '--verify' if SFT.exists() else '--output'
    run_script('build_qwen_sft.py', option, SFT)
    sft_summary = json.loads((SFT / 'summary.json').read_text())
    if sft_summary['rows'] != 4000 or sft_summary['dataset_sha256'] != MAC['dataset_sha256']:
        raise ValueError('Le corpus SFT doit correspondre au corpus de référence Mac.')
else:
    print('Mode off.')


## 2. Mesurer le petit apprentissage

La chauffe précède les trois mises à jour chronométrées. Les scripts vérifient les gradients, l'empreinte de la base et le rechargement de l'adaptateur.


In [ ]:
if MODE == 'smoke':
    smoke_dir = RUN / 'smoke'
    run_script('run_qwen_lora_smoke.py', '--device', DEVICE, '--batch-size', 2,
               '--dataset', SFT, '--output', smoke_dir)
    smoke = json.loads((smoke_dir / 'summary.json').read_text())
    if (smoke['physical_batch_size'], smoke['accumulation_steps'], smoke['effective_batch_size']) != (2, 8, 16):
        raise ValueError('Les tailles de lot doivent correspondre à la référence Mac.')
    for key in ('dataset_sha256', 'script_sha256', 'lock_sha256', 'model_sha256', 'config_sha256'):
        if smoke[key] != MAC[key]:
            raise ValueError(f'Empreinte différente de la référence Mac pour {key}.')
    if smoke['sample_rows'] != {'tsac': 32, 'madar': 32} or not smoke['reload_generation_ids_equal']:
        raise ValueError('Essai de référence incomplet.')


## 3. Vérifier les exemples les plus longs

Un passage avant et arrière utilise les deux séquences les plus longues. Cette vérification mesure le coût du lot sans effectuer de mise à jour.


In [ ]:
if MODE == 'smoke':
    capacity_path = RUN / 'capacity.json'
    run_script('check_qwen_lora_capacity.py', '--device', DEVICE, '--dataset', SFT,
               '--smoke', smoke_dir, '--output', capacity_path)
    capacity = json.loads(capacity_path.read_text())
    if capacity['sequence_lengths'] != [370, 348] or capacity['dataset_sha256'] != smoke['dataset_sha256']:
        raise ValueError('Le contrôle de capacité doit utiliser le même corpus.')


## 4. Conserver les mesures

L'archive regroupe les synthèses et le résultat de capacité. Sur Kaggle, télécharger ce ZIP depuis **Output** avant de terminer la session.


In [ ]:
if MODE == 'smoke':
    step = smoke['post_warmup_step_seconds']
    comparison = {
        'commit': os.environ.get('TUNISCOPE_COMMIT'),
        'device': smoke['device'],
        'gpu_name': capacity.get('gpu_name'),
        'seconds_per_update': step,
        'mac_reference_seconds_per_update': MAC['post_warmup_step_seconds'],
        'mac_reference_time_divided_by_current': MAC['post_warmup_step_seconds'] / step,
        'estimated_two_epoch_training_minutes': 2 * smoke['estimated_epoch_seconds'] / 60,
        'longest_batch_passed': capacity['forward_backward_passed'],
    }
    export_dir = Path(os.environ.get('TUNISCOPE_LORA_EXPORT_DIR', str(RUN))).resolve()
    export_dir.mkdir(parents=True, exist_ok=True)
    archive = export_dir / f"tuniscope_lora_{smoke['device']}_{RUN.name}.zip"
    with ZipFile(archive, 'x', compression=ZIP_DEFLATED) as bundle:
        bundle.write(SFT / 'summary.json', 'sft_summary.json')
        bundle.write(smoke_dir / 'summary.json', 'smoke_summary.json')
        bundle.write(capacity_path, 'capacity.json')
        bundle.writestr('comparison.json', json.dumps(comparison, indent=2) + '\n')
    print(json.dumps(comparison, indent=2))
    print('Archive à télécharger', archive)
